# B4: Predicting severe disease

Notebook B3 looked at the representations. This one uses them: a **prediction task**.

> Given everything recorded about a person before an index date, will they be diagnosed
> with a severe condition in the next five years?

By the end you should be able to:

1. Build the trivial baselines *before* looking at any model output.
2. Evaluate models with average precision (+ compare models)
3. Audit/interpret the quality of predictions
4. Train a classification head on a frozen encoder.
5. (If we have time) read an attribution for one person.
   1. for this part you need to install `captum` package
   2. with `uv`, run `uv add captum` to add it to the existing environment



## Setup

The 50,000-patient extract from notebook B1 and the 959K-parameter encoder from notebook B2. Nothing downloads a large file and nothing trains.

In [ ]:
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt

pd.set_option("display.width", 140)

# Works locally and on Colab.
DATA = "https://raw.githubusercontent.com/carlomarxdk/workshop-transformers/main/data/derived/workshop"
MODEL = "https://raw.githubusercontent.com/carlomarxdk/workshop-transformers/main/models/synthea-bert"
try:
    pd.read_parquet("../data/derived/workshop/cohort.parquet", columns=["patient_id"])
    DATA, MODEL = "../data/derived/workshop", "../models/synthea-bert"
except (FileNotFoundError, OSError):
    pass

import sys

sys.path.insert(0, "../scripts")
from event_bert import EventBertForMaskedLM

print("reading from:", DATA)

## 1. The task

`cohort.parquet` carries the label, the split, and the tabular attributes. Only **eligible**
patients count: those with enough history before the index date and no severe diagnosis
already on record.

The split is the one from notebook B1.


In [ ]:
vocabulary = pd.read_csv(f"{DATA}/vocabulary.csv")
sequences = pd.read_parquet(f"{DATA}/sequences.parquet")
cohort = pd.read_parquet(f"{DATA}/cohort.parquet")

df = cohort[cohort.eligible].merge(sequences, on="patient_id").reset_index(drop=True)
train = (df.split == "train").values
test = (df.split == "test").values
y = df.severe_within_5y.values.astype(int)

BASE_RATE = y[train].mean()
print(f"{len(df):,} eligible patients :  train {train.sum():,}   test {test.sum():,}")
print(
    f"positives in test: {y[test].sum()}  |  base rate of a rare disease {BASE_RATE:.3%}"
)

### Why the base rate is the first thing on screen

**5.7% positive.**

Accuracy is useless: always predicting "no" scores 94.3%, so we do not report it again.

We report one number, **average precision** (AP), the area under the precision–recall curve.
Its floor is the base rate itself, **0.057 here**, so a model at AP 0.165 finds about **2.9x**
more positives than chance among the people it ranks highest. 
**Note:** I also include **lift** (AP divided by the base rate of the group it was computed on).

## 2. Baselines first

A baseline shows what an unimpressive model scores, which is the only way to tell whether
ours has learned anything. Three of them:

1. **Record volume**: how many events the person already has.
2. **Income**: a social variable that would carry real weight in a register.
3. **Age + record volume**:  the two together. This is the number every model in this
   notebook has to beat.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

PREDICTIONS = {}  # name -> predicted probability on the test set


def fit_and_score(name, X):
    """Fit logistic regression on train, store the test probabilities, return the test AP."""
    X = np.asarray(X, dtype=np.float64)
    scaler = StandardScaler().fit(X[train])
    clf = LogisticRegression(max_iter=4000).fit(scaler.transform(X[train]), y[train])
    p = clf.predict_proba(scaler.transform(X[test]))[:, 1]
    PREDICTIONS[name] = p
    return average_precision_score(y[test], p)

In [ ]:
rows = {}
for column in ["n_events_before_index", "income", "age_at_index"]:
    rows[f"{column} alone"] = fit_and_score(f"{column} alone", df[[column]].values)

# Age and record volume together: the baseline everything else is measured against.
# (`index_date` is the cut-off; every feature here is counted strictly before it.)
rows["age + volume"] = fit_and_score(
    "age + volume", df[["age_at_index", "n_events_before_index"]].values
)

baselines = pd.DataFrame({"AP": rows, "lift": pd.Series(rows) / y[test].mean()})
baselines.round(3)

1. **Income alone is at chance.** In a real register income would carry a great deal; in
   Synthea it is drawn independently of the disease modules, so it carries nothing. A warning
   about assuming which variables matter, and a theme we return to at the end.
2. **Record volume alone beats chance.**
3. **Age is the strongest single column**, and age plus volume is stronger still.

Now see whether the transformer can beat them.

#### Code for Error bars

Error bars are routinely left out of ML papers, and they are what makes two models comparable. The snippet below computes them by bootstrapping.


In [ ]:
B = 1000
rng = np.random.default_rng(0)
n_test = int(test.sum())
RESAMPLES = [rng.integers(0, n_test, n_test) for _ in range(B)]
RESAMPLES = [
    i for i in RESAMPLES if y[test][i].sum() > 5
]  # keep resamples with positives
y_test = y[test]


def interval(values, lo=2.5, hi=97.5):
    return np.percentile(values, lo), np.percentile(values, hi)


def evaluate(name):
    """Point estimate and 95% interval for one stored model."""
    p = PREDICTIONS[name]
    aps = np.array([average_precision_score(y_test[i], p[i]) for i in RESAMPLES])
    lo, hi = interval(aps)
    return pd.Series(
        {"AP": average_precision_score(y_test, p), "AP lo": lo, "AP hi": hi},
        name=name,
    )


print(f"{len(RESAMPLES)} bootstrap resamples of {n_test:,} patients")
evaluate("age + volume").round(3)

## 3. Person summaries

Now we move back to Synthea-BERT: 
1. We pass the sequence to the transformer,
2. Average its output (mean-pool the hidden states over the tokens) and we have a vector per life.

*(About a minute locally, a few minutes on Colab's CPU. It is the only slow cell here.)*

In [ ]:
model = EventBertForMaskedLM.from_pretrained(MODEL).eval()
HIDDEN = model.bert.config.hidden_size
MAX_LEN = 128

token_id = dict(zip(vocabulary.token, vocabulary.token_id))
name_of = dict(zip(vocabulary.token_id, vocabulary.token))
CLS, SEP = token_id["[CLS]"], token_id["[SEP]"]
age_of = dict(zip(cohort.patient_id, cohort.age_at_index))

print(
    f"{model.bert.config.num_hidden_layers} transformer layers, hidden size {HIDDEN}, "
    f"{sum(p.numel() for p in model.parameters()):,} parameters, all frozen"
)

In [ ]:
def build_one(row, max_len=MAX_LEN):
    """One patient as the model sees them: ids, mask, and both clocks."""
    base = float(age_of[row.patient_id])
    days = list(row.days_before_index)
    ids = [CLS, *row.background, SEP, *row.tokens, SEP][:max_len]
    pad = len(row.background) + 2  # background and separators are timeless
    ages = ([base] * pad + [base - d / 365.25 for d in days] + [base])[:max_len]
    clock = ([0.0] * pad + [float(d) for d in days] + [0.0])[:max_len]
    return (
        torch.tensor([ids]),
        torch.ones(1, len(ids), dtype=torch.long),
        torch.tensor([ages], dtype=torch.float),
        torch.tensor([clock], dtype=torch.float),
    )


def person_vectors(
    rows, keep_last=None, background=True, batch=512, return_tokens=False
):
    """Final-layer hidden states, mean-pooled into one vector per person.

    keep_last     -- keep only the final K events (None = the whole history, 0 = none)
    background    -- include the six background tokens at the head of the sequence
    return_tokens -- also return the per-token states and their mask, which section 4 needs
                     in order to learn its own pooling. float16, so 40K x 128 x 128 is 1.3 GB
    """
    pooled = np.zeros((len(rows), HIDDEN), dtype=np.float32)
    states = (
        np.zeros((len(rows), MAX_LEN, HIDDEN), np.float16) if return_tokens else None
    )
    masks = np.zeros((len(rows), MAX_LEN), dtype=bool) if return_tokens else None

    records = list(rows.itertuples())
    for start in range(0, len(records), batch):
        chunk = records[start : start + batch]
        ids_batch, ages_batch, days_batch = [], [], []

        for r in chunk:
            tokens, days = list(r.tokens), list(r.days_before_index)
            if keep_last is not None:
                # Careful: tokens[-0:] returns the WHOLE list, not an empty one.
                cut = len(tokens) - keep_last
                tokens, days = (tokens[cut:], days[cut:]) if keep_last > 0 else ([], [])

            base_age = float(age_of.get(r.patient_id, 50.0))
            head = list(r.background) if background else []
            ids_batch.append([CLS, *head, SEP, *tokens, SEP][:MAX_LEN])
            pad = len(head) + 2
            ages_batch.append(
                ([base_age] * pad + [base_age - d / 365.25 for d in days] + [base_age])[
                    :MAX_LEN
                ]
            )
            days_batch.append(
                ([0.0] * pad + [float(d) for d in days] + [0.0])[:MAX_LEN]
            )

        width = max(len(s) for s in ids_batch)
        I = torch.zeros(len(chunk), width, dtype=torch.long)
        M = torch.zeros(len(chunk), width, dtype=torch.long)
        A_ = torch.zeros(len(chunk), width)
        D = torch.zeros(len(chunk), width)
        for j, (ids, ages, days) in enumerate(zip(ids_batch, ages_batch, days_batch)):
            n = len(ids)
            I[j, :n] = torch.tensor(ids)
            M[j, :n] = 1
            A_[j, :n] = torch.tensor(ages, dtype=torch.float)
            D[j, :n] = torch.tensor(days, dtype=torch.float)

        with torch.no_grad():
            hidden = model(
                input_ids=I,
                attention_mask=M,
                ages=A_,
                days=D,
                output_hidden_states=True,
            ).hidden_states[-1]

        mask = M.unsqueeze(-1).float()
        stop = start + len(chunk)
        pooled[start:stop] = ((hidden * mask).sum(1) / mask.sum(1)).numpy()
        if return_tokens:
            states[start:stop, :width] = hidden.numpy().astype(np.float16)
            masks[start:stop, :width] = M.numpy().astype(bool)

        if (start // batch) % 20 == 0 or stop == len(records):
            print(f"  {stop:,}/{len(records):,} people embedded")

    return (pooled, states, masks) if return_tokens else pooled


# One pass over everyone. `X` is the person summary; `H` and `MASK` are the token states
# section 4 pools for itself, and we delete them again as soon as it is done.
X, H, MASK = person_vectors(df, return_tokens=True)
print(
    f"\nX {X.shape} = (people, hidden)   H {H.shape} = (people, tokens, hidden), "
    f"{H.nbytes / 1e9:.2f} GB"
)

#### Person embeddings + Logistic Regression

Take these person embeddings and fit a logistic regression on top of them.

In [ ]:
fit_and_score("person summary", X)
pd.DataFrame([evaluate("age + volume"), evaluate("person summary")]).round(3)

One vector per life, straight into a logistic regression. It lands in the region of the baselines.

## 4. A classification head on the frozen encoder

So far the encoder's output went into `LogisticRegression`. Now we finetune Synthea-BERT: we add a head on top of the frozen model and train it for the task.

<details>
<summary><b>Optional read on the architecture</b></summary>

**Why not `BertForSequenceClassification`?** Because our model is not a plain BERT. Two things
get in the way. HuggingFace builds its own input embeddings from `input_ids`, and ours adds two
**Time2Vec clocks** on top of them, so the `ages` and `days` arguments have nowhere to go. And
its classifier reads the `[CLS]` position through a tanh pooler, while we want a pooling layer
we can inspect. Twenty lines is cheaper than fighting the abstraction, and you can read all of
them.

**The pooling layer.** Averaging the token states treats a routine dental visit and a first
cancer diagnosis as equally informative. **Attention pooling** replaces the average with a
weighted average and learns the weights:

$$w_i = \frac{\exp(h_i \cdot c)}{\sum_j \exp(h_j \cdot c)}, \qquad z = \sum_i w_i h_i$$

A single learned **context vector** $c$ scores every token by dot product, and the softmax of
those scores is the weighting. We initialise $c$ at **zero**, which makes every score equal, so
the layer *starts* as exactly mean pooling and can only move away from it if that helps.
(Initialised randomly it starts worse than the average and has to climb back; we measured it.)

</details>
</br>

|  | |
|---|---|
| **Encoder** | 959K parameters, **frozen** -> `requires_grad = False`, still inside the model |
| **Pooling** | one 128-dimensional context vector, zero-initialised. 128 parameters |
| **Head** | `128 -> 128 -> 64 -> 1`, each hidden layer followed by LayerNorm, SiLU and
dropout. About 25K parameters |

**Important notes:**
1. Our encoder has 4 layers and 959K parameters, trained on 106K synthetic patients. 
2. Real fine-tuning happens at a hundred times this size
3. However, this demo provides the pipeline:  freeze, pool, train a head, select on validation, compare against the baselines.


In [ ]:
import copy as _copy

import torch.nn as nn


class EventBertForSequenceClassification(nn.Module):
    """Frozen encoder -> attention pooling -> MLP head. One model, one forward pass."""

    def __init__(self, encoder, widths=(128, 64), dropout=0.1):
        super().__init__()
        self.encoder = encoder
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False  # frozen, but still part of the graph

        hidden = encoder.bert.config.hidden_size
        self.context = nn.Parameter(
            torch.zeros(hidden)
        )  # zero => starts as mean pooling

        # LayerNorm *after* each linear layer, then SiLU (swish). Depth, width, activation
        # and normalisation placement were all chosen on the validation slice.
        layers, size = [], hidden
        for width in widths:
            layers += [
                nn.Linear(size, width),
                nn.LayerNorm(width),
                nn.SiLU(),
                nn.Dropout(dropout),
            ]
            size = width
        layers.append(nn.Linear(size, 1))
        self.head = nn.Sequential(*layers)

    def pool(self, states, mask):
        scores = states @ self.context / self.context.numel() ** 0.5
        weights = scores.masked_fill(~mask, torch.finfo(states.dtype).min).softmax(-1)
        return (weights.unsqueeze(-1) * states).sum(1), weights

    def from_states(self, states, mask):
        """The fast path: the frozen encoder already ran, so start from its output."""
        pooled, weights = self.pool(states, mask)
        return self.head(pooled).squeeze(-1), pooled, weights

    def forward(self, input_ids, attention_mask, ages, days):
        """The full path: ids in, one risk score out. This is what section 8 attributes."""
        states = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            ages=ages,
            days=days,
            output_hidden_states=True,
        ).hidden_states[-1]
        return self.from_states(states, attention_mask.bool())[0]


# Split the TRAINING patients into fit / validation. The test set is not involved.
rng_split = np.random.default_rng(0)
train_rows = np.where(train)[0]
rng_split.shuffle(train_rows)
n_val = int(0.2 * len(train_rows))
val_rows, fit_rows = train_rows[:n_val], train_rows[n_val:]
test_rows = np.where(test)[0]
targets = torch.from_numpy(y.astype(np.float32))

example = EventBertForSequenceClassification(model)
trainable = sum(p.numel() for p in example.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in example.parameters() if not p.requires_grad)
print(f"fit {len(fit_rows):,}   validation {len(val_rows):,}   test {len(test_rows):,}")
print(
    f"{trainable:,} trainable parameters, {frozen:,} frozen "
    f"({trainable / (trainable + frozen):.1%} of the model)"
)

In [ ]:
def batch_states(index):
    """float16 in memory, float32 in the model. Casting late keeps the softmax stable."""
    return torch.from_numpy(H[index]).float(), torch.from_numpy(MASK[index])


@torch.no_grad()
def scores_for(clf, index, batch=2048):
    clf.eval()
    return np.concatenate(
        [
            clf.from_states(*batch_states(index[start : start + batch]))[0].numpy()
            for start in range(0, len(index), batch)
        ]
    )


def train_head(loss_fn, epochs=60, batch=256, lr=3e-4, seed=0):
    """Train pooling + head on the frozen encoder's output. Best-on-validation wins."""
    torch.manual_seed(seed)
    clf = EventBertForSequenceClassification(model)
    # Only the parameters that are still trainable go to the optimiser.
    optimiser = torch.optim.AdamW(
        [p for p in clf.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4
    )
    history, best = [], {"val AP": -1.0}

    for epoch in range(1, epochs + 1):
        clf.train()
        order = np.random.default_rng(epoch).permutation(fit_rows)
        running = 0.0
        for start in range(0, len(order), batch):
            rows_ = order[start : start + batch]
            optimiser.zero_grad()
            logits, _, _ = clf.from_states(*batch_states(rows_))
            loss = loss_fn(logits, targets[rows_])
            loss.backward()
            optimiser.step()
            running += loss.item() * len(rows_)

        val_ap = average_precision_score(y[val_rows], scores_for(clf, val_rows))
        history.append({"epoch": epoch, "loss": running / len(order), "val AP": val_ap})
        # Model selection on VALIDATION. The test set is not consulted.
        if val_ap > best["val AP"]:
            best = {
                "val AP": val_ap,
                "epoch": epoch,
                # The encoder never changes, so only the trained parts are worth copying.
                "state": _copy.deepcopy(
                    {
                        k: v
                        for k, v in clf.state_dict().items()
                        if not k.startswith("encoder.")
                    }
                ),
            }

    clf.load_state_dict(best["state"], strict=False)
    return clf, pd.DataFrame(history), best


classifier, history, best = train_head(nn.BCEWithLogitsLoss())
PREDICTIONS["encoder + pooling + head"] = scores_for(classifier, test_rows)
print(
    f"best epoch {best['epoch']} of {len(history)}   validation AP {best['val AP']:.3f}"
)

In [ ]:
curve = history.set_index("epoch")

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.2), dpi=120)
left.plot(curve.index, curve.loss, color="#1E152A", lw=1.6)
left.set_title("training loss", loc="left", fontsize=10)
right.plot(curve.index, curve["val AP"], color="#5AB1BB", lw=1.6)
right.axvline(best["epoch"], color="#D36582", ls="--", lw=1.2)
right.annotate(
    f"chosen: epoch {best['epoch']}",
    (best["epoch"], right.get_ylim()[0]),
    color="#D36582",
    fontsize=8,
    ha="left",
    va="bottom",
    xytext=(4, 2),
    textcoords="offset points",
)
right.set_title(
    "validation AP  (this is what picks the epoch)", loc="left", fontsize=10
)
for ax in (left, right):
    ax.set_xlabel("epoch")
    ax.grid(color="#DCDCD8", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

### What the pooling layer attends to

Our head has a context vector: it gives every token a weight. Those weights are not an explanation of the
prediction, but they do say which parts of a life the summary is built from. A
weight above `1 / n_tokens` means the head moved that token *above* what a plain average
would have given it.

In [ ]:
def pooling_weights(row, k=10):
    """The tokens this person's summary is mostly made of."""
    ids, mask, ages, clock = build_one(row)
    with torch.no_grad():
        states = classifier.encoder(
            input_ids=ids,
            attention_mask=mask,
            ages=ages,
            days=clock,
            output_hidden_states=True,
        ).hidden_states[-1]
        _, _, weights = classifier.from_states(states, mask.bool())
    out = pd.DataFrame(
        {
            "token": [name_of[int(t)] for t in ids[0]],
            "weight": weights[0].numpy().round(5),
        }
    )
    print(f"{len(out)} tokens; a plain average would give each one {1 / len(out):.5f}")
    return out.sort_values("weight", ascending=False).head(k)


held_out = df[test].reset_index(drop=True)
p_model = PREDICTIONS["encoder + pooling + head"]
riskiest = held_out.iloc[int(np.argmax(p_model))]  # <- YOU CAN PICK YOUR PATIENT HERE
print(
    f"the highest-risk held-out patient (predicted {p_model.max():.3f}), "
    f"actually diagnosed: {bool(riskiest.severe_within_5y)}"
)
pooling_weights(riskiest)

In [ ]:
# `H` is 1.3 GB and nothing below needs it. Free it before the plots and the attribution.
POOLED = np.concatenate(
    [
        classifier.from_states(
            *batch_states(np.arange(start, min(start + 2048, len(df))))
        )[1]
        .detach()
        .numpy()
        for start in range(0, len(df), 2048)
    ]
)
del H, MASK
print(f"POOLED {POOLED.shape} — the learned person summary, kept for sections 6 and 10")

### What the head bought, and what it did not

**Against the spreadsheet, the model wins.** dAP **+0.024 [+0.002, +0.047]** on 1,000 paired
resamples: AP goes from 0.140 to 0.165, a lift of 2.9x over the base rate against 2.5x. It is a small win in absolute terms, and section 7 is where we
find out who paid for it.


> **The transferable lesson.** When the encoder is frozen, precompute its output and never run
> it again. That turns a slow fine-tuning job into a fast tabular one, and the extra epochs you
> can then afford are worth more than the layers you did not unfreeze. However, if you have resources you would usually finetune the whole model.

## 5. Everything on one chart

Point estimate and 95% interval for average precision, against the base-rate floor.

In [ ]:
ORDER = [
    "income alone",
    "n_events_before_index alone",
    "age_at_index alone",
    "age + volume",
    "person summary",
    "encoder + pooling + head",
]
ORDER = [n for n in ORDER if n in PREDICTIONS]
summary = pd.DataFrame([evaluate(n) for n in ORDER])
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
pos = np.arange(len(summary))
ax.errorbar(
    summary.AP,
    pos,
    xerr=[summary.AP - summary["AP lo"], summary["AP hi"] - summary.AP],
    fmt="o",
    color="#1E152A",
    ecolor="#6B6B6B",
    capsize=3,
    lw=1.2,
)
ax.axvline(BASE_RATE, color="#D36582", ls="--", lw=1.2)
ax.set_ylim(-1.0, len(summary) - 0.4)
ax.annotate(
    f"base rate {BASE_RATE:.3f}  (AP floor)",
    (BASE_RATE, -0.85),
    color="#D36582",
    fontsize=8,
    ha="left",
    va="center",
    xytext=(5, 0),
    textcoords="offset points",
)
ax.set_yticks(pos, summary.index, fontsize=9)
ax.set_xlabel("average precision (95% bootstrap interval)")
ax.set_title("Severe disease within 5 years — held-out patients", loc="left", pad=12)
ax.grid(axis="x", color="#DCDCD8", lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

This notebook is a demo. Transformers usually need bigger scale (as compared to what we are using in this workshop). That is when the gain shows up.

## 6. What does a person embedding look like?

Every number so far has been a score. Before going further, look at the thing being scored:
7,965 held-out patients, each a 128-dimensional vector, squeezed to two dimensions the same
way notebook B3 squeezed the token space.

Same points, three colourings. The question is which of them the space is organised by.

In [ ]:
import pacmap
from scipy.stats import pearsonr

held = df[test].reset_index(drop=True)
y_held = y[test]


def project(Z, seed=7):
    return pacmap.PaCMAP(
        n_components=2, n_neighbors=15, random_state=seed
    ).fit_transform(Z)


def show(XY, colourings, title):
    """One panel per colouring, plus the correlation of each coordinate with each variable."""
    fig, axes = plt.subplots(
        1, len(colourings), figsize=(4.5 * len(colourings), 4.2), dpi=120
    )
    for ax, (label, colour) in zip(np.atleast_1d(axes), colourings.items()):
        colour = np.asarray(colour)
        if colour.dtype == object or colour.dtype.kind in "OUSb":
            for level in pd.unique(colour):
                ax.scatter(*XY[colour == level].T, s=3, lw=0, label=str(level))
            ax.legend(frameon=False, fontsize=7, markerscale=2, loc="upper left")
        else:
            ax.scatter(*XY.T, s=3, c=colour, cmap="viridis", lw=0)
        ax.set_title(label, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        for side in ax.spines.values():
            side.set_visible(False)
    fig.suptitle(title, y=1.02, fontsize=11)
    plt.tight_layout()
    plt.show()

    print("correlation of each PaCMAP coordinate with:")
    for label, v in colourings.items():
        v = np.asarray(v)
        if v.dtype.kind not in "fiub":
            continue
        print(
            f"   {label:<28} x {pearsonr(XY[:, 0], v)[0]:+.2f}   "
            f"y {pearsonr(XY[:, 1], v)[0]:+.2f}"
        )


XY = project(X[test])
show(
    XY,
    {
        "age at index": held.age_at_index.values,
        "log record volume": np.log1p(held.n_events_before_index.values),
        "severe disease within 5y": y_held,
    },
    "The same 7,965 mean-pooled person summaries, coloured three ways",
)

### The space is organised by age and record volume, not by illness

Read the correlations under the plot rather than squinting at the panels. Both coordinates
track **age** (+0.57 and +0.41), one of them also tracks **record volume** (+0.51), and
neither tracks the **outcome** (+0.09, +0.06).

The third panel makes the same point by looking like confetti. The 454 patients who go on to
a severe diagnosis are not a region of this space. They are sprinkled through it, slightly
denser where age and volume are both high.

That is the visual form of the result from section 3. A frozen encoder
trained to fill in masked events organises people by *how much health system they have used
and how old they are*, because that is what predicts the next event. Whether they will be
diagnosed with something severe in the next five years is a much weaker signal, and it does
not get its own direction.

If your prediction target had its own region here, the AP would not be 0.17.

### ✏️ Exercise 1: what else is in there?

`show()` takes any dictionary of colourings, so you can ask what else the space has encoded.
Try `sex`, `race`, `marital`, the income bin, anything in `held`. Two questions:

1. Does the attribute form **regions**, or is it sprinkled evenly? A region means the encoder
   has represented it, whether or not anyone asked it to.
2. If it does form regions, is that a problem? Age and record volume forming regions is
   unremarkable. A protected attribute forming a clean region in a representation you are
   about to predict from is the thing section 7 goes looking for.

Then re-run the projection on **`POOLED[test]`**: the summaries the pooling layer learned
for the task, and see whether training moved anything.



In [ ]:
# Your turn. Anything in `held` works as a colouring.
# show(XY, {"sex": held.sex.values, "race": held.race.values}, "Mean-pooled summaries")

# show(project(POOLED[test]), {...}, "Learned pooling")

## 7. Who is the model worst for?

One number over 7,965 people hides a lot. A model at 2.9x lift overall can be at 3.5x for one
group and 1.2x for another, and the people it fails are rarely a random sample.

This is the check to run before anyone deploys anything, and it is two lines of pandas.

**One trap first.** AP cannot be compared across groups directly, because its floor is the
group's own base rate. A group where 9% of people get sick will score a higher AP than a group
where 2% do, for a model that is equally good at both. The `lift` column divides AP by that
group's base rate, which is what makes the rows comparable (read that column, not `AP`).

In [ ]:
def stratified(name, column, min_n=200, min_positives=15):
    """Score one stored model separately within each level of `column`."""
    p = PREDICTIONS[name]
    rows = []
    for level, index in held.groupby(column, observed=True).groups.items():
        i = np.asarray(index)
        if len(i) < min_n or y_held[i].sum() < min_positives:
            continue  # too small to say anything about
        ap = average_precision_score(y_held[i], p[i])
        rows.append(
            {
                "group": str(level),
                "n": len(i),
                "positives": int(y_held[i].sum()),
                "base rate": y_held[i].mean(),
                "AP": ap,
                "lift": ap / y_held[i].mean(),
            }
        )
    return pd.DataFrame(rows).set_index("group").round(3)


# Two derived columns to stratify on, since age and volume are continuous.
held["age band"] = pd.cut(
    held.age_at_index, [29, 40, 50, 61], labels=["30-39", "40-49", "50-60"]
)
held["record volume"] = pd.qcut(
    held.n_events_before_index, 4, labels=["fewest", "few", "many", "most"]
)

SCORED = "encoder + pooling + head"
overall = evaluate(SCORED)
print(f"overall: AP {overall.AP:.3f}   lift {overall.AP / BASE_RATE:.2f}x\n")
stratified(SCORED, "sex")

### ✏️ Exercise 2: find the group the model fails

`stratified()` takes any column of `held`. Run it on **`"age band"`** and **`"record volume"`**
first, then on `"race"`, `"marital"` and anything else you can justify.

Two questions for each table:

1. Is any group's **lift** far below the overall 2.9x? That is the model working less well for
   those people than the headline number claims.
2. Where a group's raw **AP** looks high, check its base rate before believing it. A high AP in
   a high-prevalence group can be worse work than a low AP in a rare one.

Write down which group you would refuse to deploy this model for, and why.

In [ ]:
# Your turn. `stratified(SCORED, "...")` for each column you want to check.
display(stratified(SCORED, "age band"))
display(stratified(SCORED, "record volume"))

# display(stratified(SCORED, "race"))
# display(stratified(SCORED, "marital"))

<details>
<summary><b>What those two tables show</b></summary>

**Age**. Overall our model finds 2.9x more positives than chance. Inside the three age bands the lift is 2.6x, 2.8x and 2.0x, and the
oldest band, where most of the positives are, is the worst of the three.

**Record volume.** The lift by quartile runs 2.6x, 1.6x, 1.4x, 3.0x. For the middle
half of the population, people with an ordinary number of records, the model is barely better
than prevalence.

**This is why "who is it worst for?" and "what is it actually using?" are the same question.**
A stratified table is a fairness check and a leakage check at once. It is also the reason a
deployed risk score that looks good in aggregate can be useless in the clinic, where the
comparison is always between patients of similar age already in front of you.

On the base-rate trap: `race` shows it cleanly if you run it. The AP for `black` patients comes
out above the AP for `white` patients, which looks like the model serving that group better.
Their base rate is also higher. Read the `lift` column and most of the gap goes.

</details>

### ✏️ Exercise 3: run the same tables on race and ethnicity

## 8. (If we have time) Which events drove this prediction?

A risk score for one patient says nothing about *why*. **Attribution** answers that: it splits
a single prediction back across the inputs that produced it.

We use [Captum](https://captum.ai/)'s `LayerIntegratedGradients`, the standard recipe for a
transformer. It walks the input embeddings from a baseline (all `[PAD]`) to the real sequence
in `n_steps` steps and integrates the gradient of the output along the way, so every token gets
a signed number: positive pushed the risk up, negative pushed it down.

The whole pipeline (frozen encoder, learned pooling, head) is differentiable end to end, so
the gradient reaches individual events even though the encoder never trained.

In [ ]:
from captum.attr import LayerIntegratedGradients

In [ ]:
PAD = token_id["[PAD]"]
EXPLAINER = LayerIntegratedGradients(
    lambda ids, mask, ages, clock: classifier(
        input_ids=ids, attention_mask=mask, ages=ages, days=clock
    ),
    classifier.encoder.bert.get_input_embeddings(),
)


def token_attributions(row, n_steps=32):
    """Every position in one patient's sequence, with its signed contribution."""
    ids, mask, ages, clock = build_one(row)
    classifier.eval()
    scores = (
        EXPLAINER.attribute(
            ids,
            baselines=torch.full_like(ids, PAD),
            additional_forward_args=(mask, ages, clock),
            n_steps=n_steps,
        )
        .sum(-1)
        .squeeze(0)
        .detach()
        .numpy()
    )
    with torch.no_grad():
        risk = torch.sigmoid(classifier(ids, mask, ages, clock)).item()
    return pd.DataFrame(
        {
            "token": [name_of[int(t)] for t in ids[0]],
            "day": [int(d) for d in clock[0].tolist()],
            "attribution": scores.round(5),
        }
    ), risk


def attribute(row, k=8, n_steps=32):
    """The k tokens that moved this patient's score the most, either way."""
    table, _ = token_attributions(row, n_steps)
    table = table[~table.token.isin(["[CLS]", "[SEP]"])]
    return table.reindex(
        table.attribution.abs().sort_values(ascending=False).index
    ).head(k)


# Pick a patient. `RANK = 0` is the highest-risk person in the held-out set; try other ranks,
# or index `held_out` directly with `held_out.iloc[123]`.
by_risk = np.argsort(-p_model)
RANK = 0
person = held_out.iloc[by_risk[RANK]]

print(
    f"rank {RANK} of {len(held_out):,}   predicted risk {p_model[by_risk[RANK]]:.3f}   "
    f"actually developed severe disease: {bool(person.severe_within_5y)}"
)
attribute(person)

### The whole sequence, coloured

The table shows the extremes. This shows the shape of the whole life: every token in order,
written as `token [days before the cut-off]`, tinted by its attribution. **Green pushed the
predicted risk up, red pushed it down**, and the stronger the colour the larger the number.
Captum's `format_word_importances` does the colouring; the layout is ours, because our
"words" are event codes with dates attached.

In [ ]:
from captum.attr import visualization as viz
from IPython.display import HTML, display


def highlight(row, n_steps=32, skip_special=True):
    """Print one patient's whole sequence, tinted by each token's attribution."""
    table, risk = token_attributions(row, n_steps)
    if skip_special:
        table = table[~table.token.isin(["[CLS]", "[SEP]", "[PAD]"])]

    labels = [f"{t}&nbsp;[{d}d]" if d else t for t, d in zip(table.token, table.day)]
    strongest = np.abs(table.attribution).max() or 1.0
    tinted = viz.format_word_importances(labels, table.attribution / strongest)

    display(
        HTML(
            f"<div style='font-family:system-ui;font-size:12px;line-height:2.1'>"
            f"<p><b>predicted risk {risk:.3f}</b> &nbsp;|&nbsp; actually diagnosed: "
            f"{bool(row.severe_within_5y)} &nbsp;|&nbsp; {len(table)} tokens &nbsp;|&nbsp; "
            f"<span style='background:rgba(0,128,0,0.6)'>&nbsp;&nbsp;</span> raises risk "
            f"<span style='background:rgba(128,0,0,0.6)'>&nbsp;&nbsp;</span> lowers it</p>"
            f"<table style='border:none'><tr>{tinted}</tr></table></div>"
        )
    )


highlight(person)

### Reading one patient's attribution

Three habits keep this honest.

**A repeated event splits its attribution across positions.** If the same code appears seven
times, it gets seven medium numbers rather than one large one. Sum within a token before
comparing it against something that happened once.

**The background block is part of the explanation.** Age band, sex and income bin are input
tokens like any other, so they get attributions like any other. A large one is exactly what a
fairness reviewer would ask about, and section 7's stratified table is where you check whether
it shows up in the model's actual performance.

**It explains the model, not the patient.** A high attribution means "this token moved this
model's output", not "this event caused the disease". On Synthea it cannot mean the second
thing, because there is no disease here, only generator rules.

**Now pick your own patient.** Change `RANK` above and re-run the two cells: rank 0 is the
person the model is most worried about, and the middle of the ranking is where its reasoning is
least obvious. The cell below does one particular choice for you: the model's most *confident
mistake*. Attribution on a confident correct prediction mostly tells you what you already
believe; attribution on a confident mistake is where it earns its keep.

In [ ]:
# The model's most confident MISTAKE: highest predicted risk among patients who did not
# go on to a severe diagnosis.
negatives = np.where(held_out.severe_within_5y.values == 0)[0]
worst_miss = held_out.iloc[negatives[int(np.argmax(p_model[negatives]))]]
print(f"predicted risk {p_model[negatives].max():.3f}   actual: no severe disease")
display(attribute(worst_miss))
highlight(worst_miss)

## 9. (Optional) The same thing, averaged over patients

One patient's attribution is an anecdote. The obvious next move is to run it over many
patients and average by token type, which turns a per-person explanation into something that
reads like a coefficient table.

It is a useful summary and a **bad statistic**, and it is worth knowing exactly why before
you put one in a paper. The caveats are under the table.

In [ ]:
def average_attribution(rows, n=80, n_steps=16, seed=0):
    """Mean attribution per token type across `n` patients, with how often each occurred."""
    sample = rows.sample(min(n, len(rows)), random_state=seed)
    total, count = {}, {}
    for i, row in enumerate(sample.itertuples(), 1):
        table, _ = token_attributions(row, n_steps=n_steps)
        table = table[~table.token.isin(["[CLS]", "[SEP]"])]
        for token, score in zip(table.token, table.attribution):
            total[token] = total.get(token, 0.0) + score
            count[token] = count.get(token, 0) + 1
        if i % 20 == 0:
            print(f"  {i}/{len(sample)} patients attributed")

    out = pd.DataFrame(
        {
            "mean attribution": pd.Series(total) / pd.Series(count),
            "total attribution": pd.Series(total),
            "occurrences": pd.Series(count),
        }
    )
    return out[out.occurrences >= 10].sort_values("mean attribution", ascending=False)


ranked = average_attribution(held_out, n=80)
pd.concat([ranked.head(8), ranked.tail(8)]).round(4)

### Why this table is weaker than it looks

**Attribution is per position, not per type.** A code that appears seven times in one life
gets seven medium numbers, and averaging them mixes "appeared once and mattered" with
"appeared often and each occurrence mattered a little". The `occurrences` column is there so
you can see which case you are in.

**Signs cancel.** The same token can push risk up for one patient and down for another,
because attribution depends on everything else in the sequence. A mean near zero can mean
"irrelevant" or "strongly bidirectional", and this table cannot tell them apart.

**The baseline is a modelling choice.** Integrated gradients measures movement *from* an
all-`[PAD]` sequence. Choose a different baseline (an average patient, an empty history) and the numbers
change. There is no neutral reference.

**Frequent tokens are easier to sample.** Anything below the `occurrences` cut-off is dropped,
which biases the table towards routine care, which is exactly the part of Synthea that is most
generator-driven.

Use it to generate hypotheses, then test them with a method that has a null: a stratified
refit, an ablation, or the anchor-direction control from notebook B3 §5.

## What to take away

1. **Baselines before models, and the right baseline.** In administrative data, record volume
   is what a model has to beat, not age alone. Income here is at chance, which is a property of
   the simulator and a warning about assuming which variables carry signal.
2. **454 positives means error bars.** Almost every difference in this notebook is smaller than
   its own interval. Paired bootstrapping on the same resamples is what made that visible.
3. **Pick the metric before you see the numbers, and pick it for the decision.** At a 5.7%
   base rate, a metric that grades every pair of people equally spends almost all of its
   attention on pairs nobody would ever act on. AP, and lift over the base rate, grade the top
   of the ranking, which is the only part anyone uses.
4. **A frozen encoder plus a trained head is the move to take home, and precompute the frozen
   part.** 2.6% of the parameters trained, 60 epochs in seconds, because the encoder ran once.
   Everything you would tune next (depth, width, activation, loss, pooling) is then cheap
   enough to tune properly, on validation.
5. **One number over 7,965 people hides the result.** The overall 2.9x lift is mostly the model
   ranking older, heavier-record patients above younger, lighter-record ones. Hold either one
   fixed and it drops to 1.4x-2.8x. A stratified table is a fairness check and a leakage check
   at the same time.
6. **That is a fact about Synthea, not about life sequences.** The generator draws each patient
   from independent rule modules, so there is little sequential structure to recover. The same
   pipeline on the Danish registers behind life2vec comes out differently, which is why you run
   it rather than assuming either answer.

### Going further

- **Unfreeze the encoder.** `scripts/finetune.py` runs the same head with the encoder in the
  training loop. It costs about a hundred times more per epoch, and on this data it does not win.
- **Swap the loss.** Focal loss down-weights the easy negatives that are 94% of the rows. With
  this head it ties plain cross-entropy exactly (0.174 on validation against 0.174), which is
  worth knowing before reaching for it: at a 5.7% base rate the imbalance was not the binding
  constraint. At 0.5% it might be.
- **Split by `county` rather than at random**: fit in one group of counties, evaluate in
  another. Distribution shift is a more honest test than a random split, and the column is
  already in `cohort.parquet`.
- **Add a calibration curve.** A model that ranks well can still be badly calibrated, and
  calibration is what you would need before using a score to decide anything about a person.